In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/VoxDroid/Construction-Site-Safety-PPE-Detection.git
%cd /content/Construction-Site-Safety-PPE-Detection/Model-Training
!pip install -q ultralytics

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import glob
import random
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

from ultralytics import YOLO

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
class CFG:
    DEBUG = True                         # True = تجربة سريعة (3 epochs و 10% داتا) | False = تدريب حقيقي
    FRACTION = 0.10 if DEBUG else 1.0
    SEED = 88

    CLASSES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask',
               'NO-Safety Vest', 'Person', 'Safety Cone',
               'Safety Vest', 'machinery', 'vehicle']
    NUM_CLASSES_TO_TRAIN = len(CLASSES)

    EPOCHS = 3 if DEBUG else 50          # الأصل 200، و 50 مناسبة لـ Colab المجاني
    BATCH_SIZE = 16

    BASE_MODEL = 'yolov8s'
    BASE_MODEL_WEIGHTS = f'{BASE_MODEL}.pt'
    EXP_NAME = f'ppe_css_{EPOCHS}_epochs'

    OPTIMIZER = 'auto'
    LR = 1e-3
    LR_FACTOR = 0.01
    WEIGHT_DECAY = 5e-4
    DROPOUT = 0.025
    PATIENCE = 25
    PROFILE = False

    ROOT = '/content/Construction-Site-Safety-PPE-Detection/Model-Training'
    CUSTOM_DATASET_DIR = f'{ROOT}/Dataset'
    OUTPUT_DIR = '/content/Outputs'
    RUNS_DIR = f'{OUTPUT_DIR}/runs/detect'

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
RUN_DIR = f'{CFG.RUNS_DIR}/{CFG.BASE_MODEL}_{CFG.EXP_NAME}'
print(RUN_DIR)

In [ ]:
dict_file = {
    'train': f'{CFG.CUSTOM_DATASET_DIR}/train/images',
    'val': f'{CFG.CUSTOM_DATASET_DIR}/valid/images',
    'test': f'{CFG.CUSTOM_DATASET_DIR}/test/images',
    'nc': CFG.NUM_CLASSES_TO_TRAIN,
    'names': CFG.CLASSES,
}

DATA_YAML = os.path.join(CFG.OUTPUT_DIR, 'data.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.dump(dict_file, f)

print(open(DATA_YAML).read())

In [ ]:
for mode in ['train', 'valid', 'test']:
    n_img = len(os.listdir(f'{CFG.CUSTOM_DATASET_DIR}/{mode}/images'))
    n_lbl = len(os.listdir(f'{CFG.CUSTOM_DATASET_DIR}/{mode}/labels'))
    print(mode, 'images:', n_img, 'labels:', n_lbl)

In [ ]:
def display_image(image, print_info=True, hide_axis=False):
    if isinstance(image, str):
        img = Image.open(image)
        plt.imshow(img)
    elif isinstance(image, np.ndarray):
        image = image[..., ::-1]
        img = Image.fromarray(image)
        plt.imshow(img)
    else:
        raise ValueError('Provide a file path (str) or numpy array')

    if print_info:
        print('Type: ', type(img), '\n')
        print('Shape: ', np.array(img).shape, '\n')

    if hide_axis:
        plt.axis('off')

    plt.show()


def plot_random_images_from_folder(folder_path, num_images=20, seed=CFG.SEED):
    random.seed(seed)
    image_files = [f for f in os.listdir(folder_path)
                   if f.endswith(('.jpg', '.png', '.jpeg', '.gif'))]
    if len(image_files) < num_images:
        raise ValueError(f'Only {len(image_files)} images found, need {num_images}')

    selected_files = random.sample(image_files, num_images)

    num_cols = 5
    num_rows = (num_images + num_cols - 1) // num_cols
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8))

    for i, file_name in enumerate(selected_files):
        img = Image.open(os.path.join(folder_path, file_name))
        ax = axes[i // num_cols, i % num_cols]
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
folder_path = f'{CFG.CUSTOM_DATASET_DIR}/train/images/'
plot_random_images_from_folder(folder_path, num_images=20, seed=CFG.SEED)

In [ ]:
example_image_path = sorted(glob.glob(f'{CFG.CUSTOM_DATASET_DIR}/train/images/*'))[0]
print(example_image_path)
display_image(example_image_path)

In [ ]:
def get_image_properties(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f'Could not read image file: {image_path}')
    return {
        'width': img.shape[1],
        'height': img.shape[0],
        'channels': img.shape[2] if len(img.shape) == 3 else 1,
        'dtype': img.dtype,
    }

img_properties = get_image_properties(example_image_path)
img_properties

In [ ]:
class_idx = {str(i): CFG.CLASSES[i] for i in range(CFG.NUM_CLASSES_TO_TRAIN)}

class_stat = {}
data_len = {}
class_info = []

for mode in ['train', 'valid', 'test']:
    class_count = {CFG.CLASSES[i]: 0 for i in range(CFG.NUM_CLASSES_TO_TRAIN)}
    path = os.path.join(CFG.CUSTOM_DATASET_DIR, mode, 'labels')

    for file in os.listdir(path):
        with open(os.path.join(path, file)) as f:
            lines = f.readlines()
            for cls in set([line.split()[0] for line in lines if line.strip()]):
                class_count[class_idx[cls]] += 1

    data_len[mode] = len(os.listdir(path))
    class_stat[mode] = class_count
    class_info.append({'Mode': mode, **class_count, 'Data_Volume': data_len[mode]})

dataset_stats_df = pd.DataFrame(class_info)
dataset_stats_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, mode in enumerate(['train', 'valid', 'test']):
    sns.barplot(
        data=dataset_stats_df[dataset_stats_df['Mode'] == mode].drop(columns=['Mode', 'Data_Volume']),
        orient='v',
        ax=axes[i],
        palette='Set2',
    )
    axes[i].set_title(mode)
    axes[i].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
for mode in ['train', 'valid', 'test']:
    print(f'Image dimensions in {mode} set:')
    sizes = set()
    for file in glob.glob(os.path.join(CFG.CUSTOM_DATASET_DIR, mode, 'images', '*')):
        sizes.add(Image.open(file).size)
    print(sizes, '\n')

In [ ]:
model = YOLO(CFG.BASE_MODEL_WEIGHTS)

results = model.predict(
    source=example_image_path,
    classes=[0],
    conf=0.40,
    device=0,
    imgsz=640,
    save=False,
)

display_image(results[0].plot(), print_info=False, hide_axis=True)

In [ ]:
print('Model Architecture: ', CFG.BASE_MODEL_WEIGHTS)
print('Total Epochs: ', CFG.EPOCHS)
print('Batch Size: ', CFG.BATCH_SIZE)
print('Base Learning Rate: ', CFG.LR)
print('Number of Classes: ', CFG.NUM_CLASSES_TO_TRAIN)

In [ ]:
model = YOLO(CFG.BASE_MODEL_WEIGHTS)

In [ ]:
%%time

model.train(
    data=DATA_YAML,
    task='detect',
    imgsz=640,

    epochs=CFG.EPOCHS,
    batch=CFG.BATCH_SIZE,

    optimizer=CFG.OPTIMIZER,
    lr0=CFG.LR,
    lrf=CFG.LR_FACTOR,

    weight_decay=CFG.WEIGHT_DECAY,
    dropout=CFG.DROPOUT,
    fraction=CFG.FRACTION,

    patience=CFG.PATIENCE,
    profile=CFG.PROFILE,

    project=CFG.RUNS_DIR,
    name=f'{CFG.BASE_MODEL}_{CFG.EXP_NAME}',
    seed=CFG.SEED,

    val=True,
    amp=True,
    exist_ok=True,
    resume=False,

    device=0,
    workers=2,
    verbose=False,
)

In [ ]:
BEST_PT = f'{RUN_DIR}/weights/best.pt'
print(os.path.exists(BEST_PT), BEST_PT)

best_model = YOLO(BEST_PT)
metrics = best_model.val(data=DATA_YAML, split='test', device=0)

print('Precision:', metrics.box.mp)
print('Recall:', metrics.box.mr)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
test_imgs = glob.glob(f'{CFG.CUSTOM_DATASET_DIR}/test/images/*')
sample = random.choice(test_imgs)

res = best_model.predict(sample, conf=0.5, device=0)
display_image(res[0].plot(), print_info=False, hide_axis=True)

In [ ]:
best_model.export(
    format='onnx',
    imgsz=640,
    half=False,
    int8=False,
    simplify=False,
    nms=False,
)

In [ ]:
for name in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'BoxPR_curve.png', 'BoxF1_curve.png']:
    p = f'{RUN_DIR}/{name}'
    if os.path.exists(p):
        plt.figure(figsize=(12, 8))
        display_image(p, print_info=False, hide_axis=True)

In [ ]:
df = pd.read_csv(f'{RUN_DIR}/results.csv')
df = df.rename(columns=lambda x: x.replace(' ', ''))
df.to_csv(f'{CFG.OUTPUT_DIR}/training_log_df.csv', index=False)
df

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 15), sharex=True)

ax1.set_title('Box Loss')
ax1.plot(df['epoch'], df['train/box_loss'], label='Training box_loss', marker='o')
ax1.plot(df['epoch'], df['val/box_loss'], label='Validation box_loss', marker='o')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title('Classification Loss')
ax2.plot(df['epoch'], df['train/cls_loss'], label='Training cls_loss', marker='o')
ax2.plot(df['epoch'], df['val/cls_loss'], label='Validation cls_loss', marker='o')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3.set_title('Distribution Focal Loss')
ax3.plot(df['epoch'], df['train/dfl_loss'], label='Training dfl_loss', marker='o')
ax3.plot(df['epoch'], df['val/dfl_loss'], label='Validation dfl_loss', marker='o')
ax3.set_xlabel('Epoch')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/ppe_project'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy(BEST_PT, f'{SAVE_DIR}/best.pt')
onnx_path = BEST_PT.replace('.pt', '.onnx')
if os.path.exists(onnx_path):
    shutil.copy(onnx_path, f'{SAVE_DIR}/best.onnx')

print(os.listdir(SAVE_DIR))